In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import seaborn as sns

In [5]:
df=pd.read_csv(r"C:\Users\nairuti\Downloads\financial_advice_dataset.csv")
df.head()

,income,monthly_expenses,distance_to_college_work,travel_time_by_bus,transport_cost,bus_frequency,family_size,current_vehicle,loan_payment,savings,debt_focus,emergency_fund_status,investment_advice,car_recommendation
0,510.0,390.0,14.4,58.0,17.0,1.8,2,motorbike,140.0,0.0,Debt is manageable,Build emergency fund,Do not invest aggressively,Car not recommended
1,3240.0,2260.0,31.9,163.0,38.0,0.1,4,motorbike,0.0,622.0,Debt is manageable,Emergency fund adequate,Invest aggressively,Car recommended
2,3450.0,2120.0,10.5,47.0,21.0,1.7,1,motorbike,0.0,1215.0,Debt is manageable,Emergency fund adequate,Invest aggressively,Car recommended
3,5770.0,3070.0,26.9,133.0,46.0,0.2,4,none,0.0,1736.0,Debt is manageable,Emergency fund adequate,Invest aggressively,Car recommended
4,1890.0,1380.0,24.8,132.0,37.0,0.1,1,motorbike,0.0,485.0,Debt is manageable,Emergency fund adequate,Invest aggressively,Car recommended


In [11]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    ],
    remainder="passthrough"
)
model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ))
    ]
)


In [12]:
X = df.drop(columns=["debt_focus","emergency_fund_status","investment_advice","car_recommendation"])
y = df["car_recommendation"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
TARGETS = ["debt_focus", "emergency_fund_status", "investment_advice", "car_recommendation"]

In [7]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns
numerical_columns = X.select_dtypes(
    exclude=["object", "category"]
).columns
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        ),
        (
            "numerical",
            MinMaxScaler(),
            numerical_columns
        )
    ],
    remainder="passthrough"
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
for target in TARGETS:
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300, max_depth=None, random_state=42, n_jobs=-1,
            class_weight="balanced", )),
    ])
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
report = classification_report(y_test, y_pred, digits=3)
labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
print(cm)
print(report)

NameError: name 'ColumnTransformer' is not defined

In [46]:
feature_cols = [c for c in df.columns if c not in TARGETS]
numeric_cols = [c for c in feature_cols if c != "current_vehicle"]
targets = [
    "debt_focus",
    "emergency_fund_status",
    "investment_advice",
    "car_recommendation"
]

X = df.drop(columns=targets)

y = df[targets]
print(type(y))
print(y.columns.tolist())
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,)

<class 'pandas.core.frame.DataFrame'>
['debt_focus', 'emergency_fund_status', 'investment_advice', 'car_recommendation']


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for target in targets:
    pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=300, max_depth=None, random_state=42, n_jobs=-1,
            class_weight="balanced", 
        )),
    ])
    pipe.fit(X_train, y_train[target])
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test[target], y_pred)
    report = classification_report(y_test[target], y_pred, digits=3)
    labels = sorted(y[target].unique())
    cm = confusion_matrix(y_test[target], y_pred, labels=labels)
 
    print(f"\n=== {target} (held-out 20% test set) ===")
    print(f"Accuracy: {acc:.4f}")
    print(report)
    print("Confusion matrix (rows=actual, cols=predicted):")
    print("Labels:", labels)
    print(cm)
    cv_pipe = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("clf", RandomForestClassifier(
            n_estimators=300, max_depth=None, random_state=42, n_jobs=-1,
            class_weight="balanced",
        )),
    ])
    y_cv_pred = cross_val_predict(cv_pipe, X, y[target], cv=cv, n_jobs=-1)
    cv_report = classification_report(y[target], y_cv_pred, digits=3)
    print(f"\n--- {target}: 5-fold CV report (whole dataset) ---")
    print(cv_report)
 
    results[target] = {"accuracy": acc, "report": report, "cv_report": cv_report}
    models[target] = pipe
 
# Save all 4 trained pipelines + feature column order in one bundle
bundle = {
    "models": models,
    "feature_cols": feature_cols,
    "targets": TARGETS,
}


=== debt_focus (held-out 20% test set) ===
Accuracy: 0.9518
                        precision    recall  f1-score   support

    Debt is manageable      0.966     0.969     0.967      1626
Focus on reducing debt      0.912     0.902     0.907       574

              accuracy                          0.952      2200
             macro avg      0.939     0.936     0.937      2200
          weighted avg      0.952     0.952     0.952      2200

Confusion matrix (rows=actual, cols=predicted):
Labels: ['Debt is manageable', 'Focus on reducing debt']
[[1576   50]
 [  56  518]]

--- debt_focus: 5-fold CV report (whole dataset) ---
                        precision    recall  f1-score   support

    Debt is manageable      0.975     0.965     0.970      8237
Focus on reducing debt      0.899     0.926     0.912      2763

              accuracy                          0.955     11000
             macro avg      0.937     0.945     0.941     11000
          weighted avg      0.956     0.955 

In [17]:
print(bundle)

{'models': {'debt_focus': Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  Index(['current_vehicle'], dtype='object'))])),
                ('clf',
                 RandomForestClassifier(class_weight='balanced',
                                        n_estimators=300, n_jobs=-1,
                                        random_state=42))]), 'emergency_fund_status': Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('categorical',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                        

In [53]:
feature_cols = [c for c in df.columns if c not in TARGETS]
numeric_cols = [c for c in feature_cols if c != "current_vehicle"]
categorical_cols = ["current_vehicle"]
print("\n=== Feature importances (per target) ===")
ohe_names = models[TARGETS[0]].named_steps["preprocess"].transformers_[0][1].get_feature_names_out(categorical_cols)
all_feature_names = list(ohe_names) + numeric_cols
for target in TARGETS:
    clf = models[target].named_steps["clf"]
    importances = clf.feature_importances_
    order = np.argsort(importances)[::-1]
    top = [(all_feature_names[i], round(importances[i], 3)) for i in order[:5]]
    print(f"{target}: {top}")


=== Feature importances (per target) ===
debt_focus: [('loan_payment', np.float64(0.536)), ('savings', np.float64(0.283)), ('income', np.float64(0.049)), ('monthly_expenses', np.float64(0.044)), ('distance_to_college_work', np.float64(0.019))]
emergency_fund_status: [('loan_payment', np.float64(0.517)), ('savings', np.float64(0.334)), ('income', np.float64(0.043)), ('monthly_expenses', np.float64(0.031)), ('distance_to_college_work', np.float64(0.017))]
investment_advice: [('loan_payment', np.float64(0.324)), ('savings', np.float64(0.317)), ('monthly_expenses', np.float64(0.09)), ('income', np.float64(0.083)), ('distance_to_college_work', np.float64(0.041))]
car_recommendation: [('loan_payment', np.float64(0.189)), ('savings', np.float64(0.159)), ('current_vehicle_car', np.float64(0.141)), ('travel_time_by_bus', np.float64(0.121)), ('distance_to_college_work', np.float64(0.09))]


In [ ]:
!pip install joblib


In [54]:
import joblib
 
BUNDLE_PATH = "financial_advice_models.joblib"
 
bundle = joblib.load(BUNDLE_PATH)
models = bundle["models"]
feature_cols = bundle["feature_cols"]
targets = bundle["targets"]

c:\Users\nairuti\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator OneHotEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nairuti\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator FunctionTransformer from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\nairuti\anaconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.8.0 when using version 1.7.2. This might lea

In [55]:
def predict(new_df: pd.DataFrame) -> pd.DataFrame:
    missing = set(feature_cols) - set(new_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    out = new_df.copy()
    for target in targets:
        out[target] = models[target].predict(new_df[feature_cols])
    return out

In [56]:
new_data = pd.DataFrame([
        {
            "income": 4200.0,
            "monthly_expenses": 2600.0,
            "distance_to_college_work": 18.0,
            "travel_time_by_bus": 70.0,
            "transport_cost": 25.0,
            "bus_frequency": 0.5,
            "family_size": 3,
            "current_vehicle": "none",
            "loan_payment": 300.0,
            "savings": 900.0,
        }
    ])
predictions = predict(new_data)
print("\nPredictions:")
print(predictions[targets].to_string(index=False))



Predictions:
        debt_focus   emergency_fund_status investment_advice car_recommendation
Debt is manageable Emergency fund adequate Invest moderately    Car recommended


In [57]:
import joblib

joblib.dump(bundle, "all_models.pkl")
import os

print(os.path.abspath("all_models.pkl"))


c:\Users\nairuti\Downloads\all_models.pkl


In [58]:
models = {}

for target in targets:

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            random_state=42
        ))
    ])

    model.fit(X_train, y_train[target])

    models[target] = model